# Hands-on session 7: Universal MLIPs and fine-tuning

**MS.49900 AI for Materials Science (in-class, 75 min)**

| Practice | What you do | Time |
|---|---|---|
| 1 | Compare universal MLIPs (uMLIP) on Li₆PS₅Cl | 20 min |
| 2 | Fine-tune a small model on 42 DFT structures and compare with training from scratch | 30 min |
| 3 | Relax the structure, run NVT MD and check the structure against DFT | 20 min |

Run the cells in order. Cells marked `# You can change here` are blanks.

## Setup
Use a GPU runtime: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU. Runtime > Change runtime type > T4 GPU, then run again from the top.')

In [ ]:
# Install SevenNet and download the course repository with the Li6PS5Cl DFT data (about 1 min)
!pip install -q sevenn==0.13.0

import os
if not os.path.exists('MS49900-AI4M'):
    !git clone -q https://github.com/kwongibaek/MS49900-AI4M.git

DATA = 'MS49900-AI4M/Data'
print(sorted(f for f in os.listdir(DATA) if f.startswith(('lpsc_', 'sevennet_'))))

In [ ]:
import time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ase.io import read
from ase.units import bar
from sevenn.calculator import SevenNetCalculator

to_kbar = 1 / bar / 1000          # eV/Å³ -> kbar


def dft_labels(traj):
    """DFT energy (eV/atom), forces (eV/Å) and stress (kbar) stored in the data file."""
    E = np.array([a.get_potential_energy() / len(a) for a in traj])
    F = np.concatenate([a.get_forces().ravel() for a in traj])
    S = np.concatenate([a.get_stress().ravel() for a in traj]) * to_kbar
    return E, F, S


def predict(calc, traj):
    """MLIP energy, forces and stress for every structure, plus seconds per structure."""
    E, F, S = [], [], []
    warm = traj[0].copy(); warm.calc = calc; warm.get_forces()   # warm-up, not timed
    t0 = time.time()
    for atoms in traj:
        a = atoms.copy()          # copy keeps the DFT labels of the original
        a.calc = calc
        E.append(a.get_potential_energy() / len(a))
        F.append(a.get_forces().ravel())
        S.append(a.get_stress().ravel() * to_kbar)
    sec = (time.time() - t0) / len(traj)
    return np.array(E), np.concatenate(F), np.concatenate(S), sec


def free_gpu():
    """Release GPU memory. Delete your own reference to the model first, e.g. `del calc; free_gpu()`."""
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))

def parity(ax, x, y, label, unit):
    ax.scatter(x, y, s=3, alpha=0.3)
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
    ax.set_xlabel(f'DFT {label} ({unit})')
    ax.set_ylabel(f'MLIP {label} ({unit})')
    ax.set_aspect('equal')

## Practice 1. Compare universal MLIPs (zero-shot)

A **universal MLIP** is trained once on large DFT databases covering most of the periodic table. Larger models trained on more data rank higher on benchmarks such as [Matbench Discovery](https://matbench-discovery.materialsproject.org/). Do they also work better on *our* system?

| Model | Training data | Size |
|---|---|---|
| `7net-0` | MPtrj (1.6 M structures) | 0.8 M parameters |
| `7net-mf-ompa` | MPtrj + sAlex + OMat24 | larger, multi-fidelity |
| `7net-omni` | 15 open DFT datasets | largest, multi-task |

We test on 10 snapshots of a separate DFT MD run (`test_md.extxyz`, same DFT setting as the pre-class data). Its cell is slightly smaller (961 Å³ per unit cell) than that of the 600 K and 1200 K runs we fine-tune on in Practice 2 (965 Å³).

In [ ]:
# Settings
N_TEST = 10
MODELS = {                      # label: (model name, extra options)
    '7net-0':       ('7net-0', {}),
    '7net-mf-ompa': ('7net-mf-ompa', {'modal': 'mpa'}),
    '7net-omni':    ('7net-omni', {'modal': 'mpa'}),
}

In [ ]:
test_all = read(f'{DATA}/lpsc_test_md.extxyz.gz', ':')
test = test_all[::len(test_all) // N_TEST][:N_TEST]
dft_E, dft_F, dft_S = dft_labels(test)
print(len(test), 'test structures,', len(test[0]), 'atoms each')

**Blank 1.** The energy offset from the pre-class hides how well each model ranks structures. Add the energy RMSE after removing the offset.

In [ ]:
results, preds = {}, {}

def add_result(label, calc):
    E, F, S, sec = predict(calc, test)
# You can change here------------------------------------------------------------
    e_shift = ...   # energy RMSE after removing each mean (see pre-class)
# -------------------------------------------------------------------------------
    preds[label] = (E, F, S)
    results[label] = {
        'Energy RMSE (eV/atom)': rmse(dft_E, E),
        'Energy RMSE, offset removed (meV/atom)': 1000 * e_shift,
        'Force RMSE (eV/Å)': rmse(dft_F, F),
        'Stress RMSE (kbar)': rmse(dft_S, S),
        'Time (s/structure)': sec,
    }

for label, (name, options) in MODELS.items():
    print('Running', label, '...')
    calc = SevenNetCalculator(name, **options)
    add_result(label, calc)
    del calc; free_gpu()

pd.DataFrame(results).T.round(3)

In [ ]:
def plot_parity(labels):
    fig, axes = plt.subplots(len(labels), 3, figsize=(12, 3.8 * len(labels)), squeeze=False)
    for row, label in zip(axes, labels):
        E, F, S = preds[label]
        parity(row[0], dft_E - dft_E.mean(), E - E.mean(), 'rel. energy', 'eV/atom')
        parity(row[1], dft_F, F, 'force', 'eV/Å')
        parity(row[2], dft_S, S, 'stress', 'kbar')
        row[0].set_title(label, loc='left', fontweight='bold')
    plt.tight_layout(); plt.show()

plot_parity(list(MODELS))

**What the results show.** The larger uMLIPs (`7net-mf-ompa`, `7net-omni`) give clearly lower force errors than `7net-0` (about 0.10 vs 0.18 eV/Å) and lower offset-removed energy errors (about 3 vs 10 meV/atom), but each structure takes about 10 times longer. Their stress error, however, is larger (about 7.5 vs 5.3 kbar). A higher benchmark rank does not make every property better on our system. One likely reason is that stress is sensitive to the DFT setting, and our data were computed with a setting different from the training data of these models (the same difference causes the energy offset in the pre-class). Which model to use depends on the property you need and the time you can afford. Practice 2 tries a different route: adapt a small, fast model to our own DFT setting.

*Your numbers may differ slightly from run to run.*

If a model runs out of GPU memory, restart the runtime and remove it from `MODELS`.

## Practice 2. Fine-tune a small model

Instead of a bigger model, we adapt a small pretrained model (`7net-0-small`) to our DFT setting with a few structures, in three steps.

1. **Shift only**: fix the constant energy offset by moving the reference energies. No training, forces and stress do not change.
2. **Fine-tune**: retrain all weights from the pretrained values.
3. **From scratch**: train the same model from random weights, to see what pretraining gives.

- Data: 600 K + 1200 K snapshots, 25% for training (42 structures), 10 for validation
- Train: 10 epochs, Adam, learning rate 0.004 decreasing linearly

In [ ]:
import sevenn.util as util
from copy import deepcopy
from torch_geometric.loader import DataLoader
from sevenn.train.graph_dataset import SevenNetGraphDataset
from sevenn.train.trainer import Trainer
from sevenn.error_recorder import ErrorRecorder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0)

# Settings
TRAIN_FRACTION = 0.25
N_VALID = 10
EPOCHS = 10
BATCH_SIZE = 4
TRAIN_SETTINGS = {
    'device': device,
    'is_ddp': False,
    'optimizer': 'adam',
    'optim_param': {'lr': 0.004},
    'scheduler': 'linearlr',
    'scheduler_param': {'start_factor': 1.0, 'total_iters': EPOCHS, 'end_factor': 0.0001},
    'loss': 'huber',
    'loss_param': {'delta': 0.01},
    'force_loss_weight': 0.1,
    'stress_loss_weight': 0.01,
}

model, config = util.model_from_checkpoint(f'{DATA}/sevennet_0_small_checkpoint.pth')
config.update(TRAIN_SETTINGS)

dataset = SevenNetGraphDataset(
    cutoff=config['cutoff'], root='sevenn_data', drop_info=False,
    files=[f'{DATA}/lpsc_600K.extxyz.gz', f'{DATA}/lpsc_1200K.extxyz.gz'],
)
dataset = dataset.shuffle()
n_train = int(len(dataset) * TRAIN_FRACTION)
train_loader = DataLoader(dataset[:n_train], batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(dataset[n_train:n_train + N_VALID], batch_size=BATCH_SIZE)
print(f'{len(dataset)} structures: {n_train} train, {N_VALID} validation')

### Step 1. Shift only

Each SevenNet model ends with a reference energy per element. Moving all of them by the same amount shifts the total energy by a constant and leaves forces and stress unchanged.

**Blank 2.** Compute the offset (eV/atom) between DFT and `7net-0-small` on 10 structures from the 600 K and 1200 K files (every 20th snapshot).

*(All our structures have the same composition, so one offset for every element is enough. For datasets with different compositions, fit one offset per element by linear regression, as in the SevenNet tutorial.)*

In [ ]:
train_atoms = read(f'{DATA}/lpsc_600K.extxyz.gz', '::20') + read(f'{DATA}/lpsc_1200K.extxyz.gz', '::20')
calc = SevenNetCalculator(f'{DATA}/sevennet_0_small_checkpoint.pth')
E_pred, _, _, _ = predict(calc, train_atoms)
del calc; free_gpu()
E_dft = np.array([a.get_potential_energy() / len(a) for a in train_atoms])
# You can change here------------------------------------------------------------
offset = ...   # mean of (DFT - model) energy per atom, eV/atom
# -------------------------------------------------------------------------------
print(f'offset = {offset:.3f} eV/atom from {len(train_atoms)} structures')

shift_model, shift_config = util.model_from_checkpoint(f'{DATA}/sevennet_0_small_checkpoint.pth')
with torch.no_grad():
    shift_model._modules['rescale_atomic_energy'].shift += offset    # same offset for every element
shift_config.update(TRAIN_SETTINGS)
Trainer.from_config(shift_model, shift_config).write_checkpoint('checkpoint_shift.pth', config=shift_config, epoch=0)
del shift_model; free_gpu()

### Step 2. Fine-tune

In [ ]:
def evaluate(trainer, loader, rec):
    """Errors on a data set without updating the weights."""
    trainer.run_one_epoch(loader, is_train=False, error_recorder=rec)
    return rec.epoch_forward()

def report(epoch, va):
    print(f"epoch {epoch:2d} | valid MAE  E {va['Energy_MAE (eV/atom)']:.4f} eV/atom"
          f"  F {va['Force_MAE (eV/Å)']:.4f} eV/Å  S {va['Stress_MAE (kbar)']:.3f} kbar"
          + ('   (before training)' if epoch == 0 else ''))

trainer = Trainer.from_config(model, config)
train_rec = ErrorRecorder.from_config(config)
valid_rec = deepcopy(train_rec)

# Epoch 0: errors of the pretrained model before any training
history = {'train': [evaluate(trainer, train_loader, train_rec)], 'valid': [evaluate(trainer, valid_loader, valid_rec)]}
report(0, history['valid'][0])
for epoch in range(1, EPOCHS + 1):
    trainer.run_one_epoch(train_loader, is_train=True, error_recorder=train_rec)
    trainer.run_one_epoch(valid_loader, is_train=False, error_recorder=valid_rec)
    trainer.scheduler_step()
    tr, va = train_rec.epoch_forward(), valid_rec.epoch_forward()
    history['train'].append(tr); history['valid'].append(va)
    report(epoch, va)

trainer.write_checkpoint('checkpoint_fine_tuned.pth', config=config, epoch=EPOCHS)
free_gpu()

### Step 3. Does pretraining help?

Train the **same model on the same 42 structures for the same 10 epochs**, but start from random weights instead of the pretrained ones. Everything else is the same as in Step 2, including the element reference energies, so only the starting weights differ. Compare the force and stress errors in particular.

In [ ]:
# Same architecture, same data, same training settings, random starting weights
scratch_model, scratch_config = util.model_from_checkpoint(f'{DATA}/sevennet_0_small_checkpoint.pth')
torch.manual_seed(1)
with torch.no_grad():
    for name, p in scratch_model.named_parameters():
        if name.endswith('weight'):           # weights of the linear layers and radial networks
            torch.nn.init.normal_(p)          # e3nn layers start from N(0, 1) weights
        # kept: radial basis, normalisation constants, element reference energies and scale
scratch_config.update(TRAIN_SETTINGS)

scratch_trainer = Trainer.from_config(scratch_model, scratch_config)
rec_train = ErrorRecorder.from_config(scratch_config)
rec_valid = deepcopy(rec_train)
scratch_history = {'train': [evaluate(scratch_trainer, train_loader, rec_train)],
                   'valid': [evaluate(scratch_trainer, valid_loader, rec_valid)]}   # epoch 0: random weights
report(0, scratch_history['valid'][0])
for epoch in range(1, EPOCHS + 1):
    scratch_trainer.run_one_epoch(train_loader, is_train=True, error_recorder=rec_train)
    scratch_trainer.run_one_epoch(valid_loader, is_train=False, error_recorder=rec_valid)
    scratch_trainer.scheduler_step()
    tr, va = rec_train.epoch_forward(), rec_valid.epoch_forward()
    scratch_history['train'].append(tr); scratch_history['valid'].append(va)
    report(epoch, va)

scratch_trainer.write_checkpoint('checkpoint_scratch.pth', config=scratch_config, epoch=EPOCHS)
del scratch_trainer, scratch_model; free_gpu()

### Learning curves

Validation error after each epoch for the fine-tuned model (pretrained start) and the model trained from scratch, and the training error of the fine-tuned model. **Epoch 0** is the error before any training: for the fine-tuned model it is the pretrained `7net-0-small`, for the other model it is the random start.

Read the curves in two ways:
- **What did training improve?** Compare each curve with its own epoch 0. An error that ends where it started was not improved by fine-tuning.
- **Overfitting?** If the validation error stops falling while the training error keeps falling, the model starts to overfit.

In [ ]:
keys = [('Energy_MAE (eV/atom)', 'Energy MAE (eV/atom)'),
        ('Force_MAE (eV/Å)', 'Force MAE (eV/Å)'),
        ('Stress_MAE (kbar)', 'Stress MAE (kbar)')]
epochs = np.arange(0, EPOCHS + 1)                      # 0 = before training
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, (key, title) in zip(axes, keys):
    ax.plot(epochs, [h[key] for h in history['train']], 'o-', color='#9DB4D3', label='Fine-tuned, train')
    ax.plot(epochs, [h[key] for h in history['valid']], 's-', color='#2F5597', label='Fine-tuned, validation')
    ax.plot(epochs, [h[key] for h in scratch_history['valid']], '^--', color='#C0504D', label='From scratch, validation')
    ax.axvline(0.5, color='gray', lw=0.8, ls=':')        # training starts after epoch 0
    ax.set_xlabel('Epoch (0 = before training)'); ax.set_xticks(epochs); ax.set_title(title); ax.set_yscale('log')
axes[0].legend(fontsize=9)
plt.tight_layout(); plt.savefig('practice2_learning_curve.png', dpi=150); plt.show()

**Blank 3.** Load your fine-tuned checkpoint and compare the pretrained, shifted, from-scratch and fine-tuned small models with the uMLIPs.

In [ ]:
# You can change here------------------------------------------------------------
ft_path = ...   # path of the checkpoint you saved above
# -------------------------------------------------------------------------------

for label, path in [('7net-0-small', f'{DATA}/sevennet_0_small_checkpoint.pth'),
                    ('7net-0-small + shift', 'checkpoint_shift.pth'),
                    ('Same model, from scratch', 'checkpoint_scratch.pth'),
                    ('7net-0-small + FT', ft_path)]:
    calc = SevenNetCalculator(path)
    add_result(label, calc)
    del calc; free_gpu()

summary = pd.DataFrame(results).T
summary.round(3)

In [ ]:
cols = ['Energy RMSE (eV/atom)', 'Force RMSE (eV/Å)', 'Stress RMSE (kbar)']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, cols):
    summary[col].plot.bar(ax=ax, color=['#9DB4D3'] * (len(summary) - 1) + ['#2F5597'], logy=(col.startswith('Energy')))
    ax.set_title(col); ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.savefig('practice2_rmse.png', dpi=150); plt.show()

plot_parity(['7net-0-small', '7net-0-small + FT'])

**What the results show.**

- **Shift.** Moving the reference energies removes the 2.4 eV/atom offset without any training: the energy RMSE drops from about 2.4 to 0.014 eV/atom, which is the offset-removed error of `7net-0-small` (about 14 meV/atom). Forces and stress stay exactly the same.
- **Fine-tuning** halves the remaining energy error (offset-removed RMSE about 14 → 7 meV/atom) and cuts the stress error from about 6.0 to 1.5 kbar. It does **not** improve the forces: the test force RMSE stays at about 0.24 eV/Å, and the validation force error ends close to its epoch-0 value. Possible reasons: (a) the loss is dominated by energy and stress, and the force term has a weight of only 0.1; (b) our DFT setting differs from the pretraining data mostly in the energy reference and the stress, while the pretrained forces are already of the right size, and 42 structures for 10 epochs are not enough to refine them; (c) the high starting learning rate (0.004) first pushes the model away from its pretrained forces (the jump at epoch 2), and the remaining epochs only bring it back. A smaller learning rate alone does not help, though: with 0.001 the model cannot even remove the energy offset in 10 epochs (post-class B). A larger force weight or more structures could improve the forces; post-class B tests this.
- **Fine-tuned small model vs uMLIPs.** The large uMLIPs still give lower force errors (about 0.10 vs 0.24 eV/Å). The fine-tuned small model gives the lowest energy and stress errors and is about 20 times faster.
- **From scratch.** With the same 42 structures and 10 epochs, the model trained from random weights is much worse, mainly for forces and stress (about 0.74 eV/Å and 13 kbar), and its validation force error hardly moves from epoch 0. 42 structures are far too few to learn the interactions from nothing. Pretraining on millions of DFT structures already gives the model general knowledge of chemical bonding, so fine-tuning only has to adapt it to our DFT setting.
- **No overfitting.** The training and validation errors of the fine-tuned model fall together. The jump at epoch 2 comes from the large starting learning rate, not from overfitting.

*Your numbers may differ slightly from run to run.*

## Practice 3. Relax the structure and run MD

Now use the model as a simulator, and look at the structures.

1. **Relaxation**: LBFGS optimizer + `FrechetCellFilter` (relaxes atoms and cell volume). Compare the volume per unit cell with experiment (956–960 Å³).
2. **Check against DFT**: compare each model's energy–volume curve with DFT.
3. **NVT MD**: Langevin dynamics at 600 K for 2 ps. Plot the mean squared displacement (MSD) of each element and the paths of the Li ions.
4. **Check the MD structure**: compare the radial distribution functions with the DFT MD at 600 K.

In [ ]:
from ase.optimize import LBFGS
from ase.filters import FrechetCellFilter
from ase.visualize.plot import plot_atoms
from ase.data import atomic_numbers
from ase.data.colors import jmol_colors

ELEMENTS = ['Li', 'P', 'S', 'Cl']
ATOM_COLORS = {el: jmol_colors[atomic_numbers[el]] for el in ELEMENTS}            # for atoms
LINE_COLORS = {'Li': '#A05EE0', 'P': '#E07000', 'S': '#C9A400', 'Cl': '#1A9E1A'}   # for lines

def relax(atoms, calc, fmax=0.05, steps=1000):
    """Relax atoms and cell. Returns the relaxed structure and the volume per unit cell at each step."""
    a = atoms.copy(); a.calc = calc
    opt = LBFGS(FrechetCellFilter(a, hydrostatic_strain=True), logfile=None)
    vol = []
    opt.attach(lambda: vol.append(a.get_volume() / 8))        # 2x2x2 supercell
    opt.run(fmax=fmax, steps=steps)
    a.calc = None                                               # detach the model so its GPU memory can be freed
    return a, np.array(vol)

start = read(f'{DATA}/lpsc_test_md.extxyz.gz', 0)
volumes, traces = {}, {}
for label, path in [('7net-0', '7net-0'), ('7net-0-small + FT', ft_path)]:
    calc = SevenNetCalculator(path)
    relaxed, traces[label] = relax(start, calc)
    volumes[label] = traces[label][-1]
    print(f'{label:18s} {volumes[label]:7.1f} Å³ per unit cell ({len(traces[label]) - 1} steps)')
    if label == '7net-0-small + FT':
        relaxed_ft = relaxed
    del calc; free_gpu()
print('Experiment         956–960 Å³ per unit cell')

In [ ]:
fig = plt.figure(figsize=(15, 4.8))
for k, (a, title) in enumerate([(start, 'Start: DFT MD snapshot'), (relaxed_ft, 'Relaxed (fine-tuned model)')]):
    ax = fig.add_subplot(1, 3, k + 1)
    plot_atoms(a, ax, radii=0.5, rotation='10x,10y')
    ax.set_title(f'{title}\n{a.get_volume() / 8:.1f} Å³ per unit cell'); ax.axis('off')
ax = fig.add_subplot(1, 3, 3)
for label, vol in traces.items():
    ax.plot(vol, label=label)
ax.axhspan(956, 960, color='#2F5597', alpha=0.15, label='Experiment')
ax.set_xlabel('Optimizer step'); ax.set_ylabel('Volume per unit cell (Å³)'); ax.legend()
handles = [plt.Line2D([], [], marker='o', ls='', mfc=ATOM_COLORS[e], mec='k', ms=9, label=e) for e in ELEMENTS]
fig.legend(handles=handles, loc='lower left', ncol=4, frameon=False, bbox_to_anchor=(0.12, 0))
plt.tight_layout(rect=(0, 0.07, 1, 1)); plt.savefig('practice3_relax.png', dpi=150); plt.show()

Rotate the relaxed structure with the mouse.

In [ ]:
from ase.visualize import view
view(relaxed_ft, viewer='x3d')

### Check against DFT: energy–volume curve

The relaxed volumes are above experiment. Is that the model's error or DFT's? Compare each model with a DFT energy–volume curve of Li₆PS₅Cl (11 volumes, from the SevenNet tutorial data). Energies are shown relative to each curve's minimum, so the energy offset does not matter here. This DFT curve uses Materials Project-compatible PBE settings, like the uMLIP training data; our fine-tuning data used a different setting and only one cell volume (965 Å³).

In [ ]:
from ase.eos import EquationOfState

eos_set = read(f'{DATA}/lpsc_eos.extxyz.gz', ':')
vols = np.array([a.get_volume() / 8 for a in eos_set])                 # per unit cell
curves = {'DFT': np.array([a.get_potential_energy() / len(a) for a in eos_set])}
EOS_MODELS = {
    '7net-0': ('7net-0', {}),
    '7net-omni': ('7net-omni', {'modal': 'mpa'}),
    '7net-0-small + FT': (ft_path, {}),
}
for label, (name, options) in EOS_MODELS.items():
    calc = SevenNetCalculator(name, **options)
    curves[label] = predict(calc, eos_set)[0]
    del calc; free_gpu()

eos_v0 = {}
fig, ax = plt.subplots(figsize=(7, 4.8))
for label, E in curves.items():
    v0, e0, B = EquationOfState(vols, E, eos='birchmurnaghan').fit()
    eos_v0[label] = v0
    style = dict(color='k', lw=2.5, marker='o') if label == 'DFT' else dict(lw=1.5, marker='.')
    ax.plot(vols, (E - e0) * 1000, label=f'{label}: V0 = {v0:.0f} Å³', **style)
ax.axvspan(956, 960, color='#2F5597', alpha=0.15, label='Experiment 956–960 Å³')
ax.set_xlabel('Volume per unit cell (Å³)'); ax.set_ylabel('E − E_min (meV/atom)')
ax.legend(fontsize=9); ax.set_title('Energy–volume curve of Li₆PS₅Cl')
plt.tight_layout(); plt.savefig('practice3_eos.png', dpi=150); plt.show()

pd.DataFrame({'EOS minimum V0': pd.Series(eos_v0),
              'Relaxed (this notebook)': pd.Series(volumes)}).round(1)

In [ ]:
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary

def run_nvt(atoms, calc, T, n_steps, dt_fs=2.0, interval=10, seed=0):
    """Langevin NVT MD. Returns times (ps), positions (frames x atoms x 3), temperatures."""
    a = atoms.copy(); a.calc = calc
    MaxwellBoltzmannDistribution(a, temperature_K=T, rng=np.random.default_rng(seed))
    Stationary(a)
    dyn = Langevin(a, dt_fs * units.fs, temperature_K=T, friction=0.01 / units.fs,
                   rng=np.random.default_rng(seed + 1))
    pos, temp = [], []                 # the observer also runs at step 0
    def record():
        pos.append(a.get_positions().copy()); temp.append(a.get_temperature())
    dyn.attach(record, interval=interval)
    dyn.run(n_steps)
    t = np.arange(len(pos)) * interval * dt_fs / 1000
    return t, np.array(pos), np.array(temp)

def msd_by_element(pos, atoms):
    """MSD (Å²) of each element; the center-of-mass drift is removed."""
    m = atoms.get_masses()
    disp = pos - pos[0]
    disp -= (disp * m[None, :, None]).sum(1, keepdims=True) / m.sum()
    sym = np.array(atoms.get_chemical_symbols())
    return {el: (disp[:, sym == el] ** 2).sum(-1).mean(1) for el in ELEMENTS}

def plot_li_paths(ax, atoms, pos, title):
    """Top view of the bottom half of the cell: framework atoms at the start and Li paths."""
    sym = np.array(atoms.get_chemical_symbols())
    sel = pos[0, :, 2] < atoms.cell[2, 2] / 2
    for el in ['S', 'P', 'Cl']:
        m = (sym == el) & sel
        ax.scatter(pos[0, m, 0], pos[0, m, 1], s=40, color=ATOM_COLORS[el], edgecolors='k', linewidths=0.3, label=el)
    for i in np.where((sym == 'Li') & sel)[0]:
        ax.plot(pos[:, i, 0], pos[:, i, 1], color=LINE_COLORS['Li'], lw=0.8)
    ax.plot([], [], color=LINE_COLORS['Li'], lw=2, label='Li path')
    ax.set_aspect('equal'); ax.set_xlabel('x (Å)'); ax.set_ylabel('y (Å)'); ax.set_title(title)
    ax.legend(loc='upper right', fontsize=9)

MD_STEPS = 1000                      # 2 fs x 1000 = 2 ps
calc = SevenNetCalculator(ft_path)
t, pos, temp = run_nvt(relaxed_ft, calc, T=600, n_steps=MD_STEPS)
del calc; free_gpu()
print(f'mean temperature {temp[len(temp)//5:].mean():.0f} K')

msd = msd_by_element(pos, relaxed_ft)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for el, y in msd.items():
    axes[0].plot(t, y, label=el, color=LINE_COLORS[el])
axes[0].set_xlabel('Time (ps)'); axes[0].set_ylabel('MSD (Å²)'); axes[0].legend(); axes[0].set_title('MSD, NVT 600 K')
plot_li_paths(axes[1], relaxed_ft, pos, 'Li paths over 2 ps (bottom half of the cell)')
plt.tight_layout(); plt.savefig('practice3_msd.png', dpi=150); plt.show()

### Check against DFT: structure during MD

A low force RMSE does not guarantee a correct simulation. Compare the **radial distribution function** g(r) of your MD with that of the DFT MD at the same temperature (the 600 K snapshots). g(r) is the probability of finding an atom of type B at distance r from an atom of type A, relative to a uniform gas; its peaks are the typical bond distances. The first 20% of each run is skipped (equilibration).

In [ ]:
from ase import Atoms
from ase.data import atomic_numbers
from ase.geometry.rdf import get_rdf

PAIRS = [('P', 'S'), ('Li', 'S'), ('Li', 'Li')]
R_MAX, N_BINS = 6.0, 120                               # Å; R_MAX must stay below half the cell length

def average_rdf(frames, pair):
    """Partial g(r) of an element pair, averaged over the frames. Returns r (Å) and g(r)."""
    out = [get_rdf(a, R_MAX, N_BINS, elements=[atomic_numbers[e] for e in pair]) for a in frames]
    return out[0][1], np.mean([g for g, _ in out], axis=0)

symbols = relaxed_ft.get_chemical_symbols()
md_frames = [Atoms(symbols, positions=p, cell=relaxed_ft.cell, pbc=True) for p in pos[len(pos) // 5:]]
dft_md = read(f'{DATA}/lpsc_600K.extxyz.gz', ':')
dft_frames = dft_md[len(dft_md) // 5:]

fig, axes = plt.subplots(1, len(PAIRS), figsize=(14, 3.8))
for ax, pair in zip(axes, PAIRS):
    r, g_dft = average_rdf(dft_frames, pair)
    _, g_md = average_rdf(md_frames, pair)
    ax.plot(r, g_dft, 'k-', lw=2, label='DFT MD, 600 K')
    ax.plot(r, g_md, '-', color='#2F5597', lw=1.5, label='Fine-tuned MLIP MD, 600 K')
    ax.set_xlabel('r (Å)'); ax.set_ylabel('g(r)'); ax.set_title(f'{pair[0]}–{pair[1]}')
axes[0].legend(fontsize=9)
plt.tight_layout(); plt.savefig('practice3_rdf.png', dpi=150); plt.show()

*(Optional)* Watch your MD run in 3D, in the same style as the DFT MD movie in the pre-class. Purple: Li, orange: P, yellow: S, green: Cl; **red**: the six Li ions that moved farthest, with their paths. You should see the same picture as in DFT: Li moving between sites while the framework only shakes.

In [ ]:
# (Optional) Watch your MD run as a 3D movie (drag to rotate, scroll to zoom)
!pip install -q py3Dmol
import io, itertools, py3Dmol
from ase import Atoms
from ase.io import write

symbols = np.array(relaxed_ft.get_chemical_symbols())
cell = relaxed_ft.cell.array
li = np.where(symbols == 'Li')[0]
fast = li[np.argsort(np.linalg.norm(pos[-1, li] - pos[0, li], axis=1))[-6:]]   # the 6 Li that moved farthest
# ASE MD positions are not wrapped back into the cell, so the paths are continuous

frames = [Atoms(symbols, positions=p) for p in pos]
buf = io.StringIO(); write(buf, frames, format='xyz')

v = py3Dmol.view(width=700, height=500)
v.addModelsAsFrames(buf.getvalue(), 'xyz')
v.setStyle({'sphere': {'scale': 0.2, 'colorscheme': 'Jmol'}})
v.setStyle({'index': fast.tolist()}, {'sphere': {'scale': 0.45, 'color': 'red'}})
for i in fast:                                                         # path of each highlighted Li
    for p, q in zip(pos[:-1, i], pos[1:, i]):
        v.addCylinder({'start': dict(zip('xyz', p.tolist())), 'end': dict(zip('xyz', q.tolist())),
                       'radius': 0.07, 'color': 'red', 'fromCap': 1, 'toCap': 1})
corners = np.array(list(itertools.product([0, 1], repeat=3))) @ cell    # outline of the simulation cell
edge = np.linalg.norm(cell, axis=1)
for a, b in itertools.combinations(corners, 2):
    if np.isclose(np.linalg.norm(a - b), edge).any():
        v.addCylinder({'start': dict(zip('xyz', a.tolist())), 'end': dict(zip('xyz', b.tolist())),
                       'radius': 0.05, 'color': 'gray'})
v.zoomTo(); v.rotate(-25, 'x'); v.rotate(30, 'y')
v.animate({'loop': 'forward', 'interval': 80}); v.show()

**What the results show.**

- **MD.** Only Li moves; P, S and Cl vibrate around their sites. Most Li ions rattle around their sites, and a few hop to a neighbouring site within 2 ps. 2 ps gives a Li MSD of only a few Å², too short and noisy for a reliable diffusion coefficient; post-class A runs longer MD for that.
- **Volume.** The relaxed volumes are larger than experiment. The DFT energy–volume curve has its minimum at about 976 Å³ per unit cell, and `7net-0` and `7net-omni` reproduce it within 1–2 Å³. So the pretrained models follow DFT, and it is DFT (PBE) that overestimates the volume, by about 2% compared with experiment.
- **Fine-tuned model.** Its energy–volume curve is steeper, with its minimum near 963 Å³. Two reasons are possible: it saw only one volume (965 Å³), so away from it the curve is an extrapolation; and it learned a different DFT setting, whose equilibrium volume need not be 976 Å³. This notebook cannot separate the two; that would need DFT energy–volume data in the fine-tuning setting. Fine-tuning data should cover the conditions you want to simulate (volumes, temperatures, defects).
- **RDF.** The P–S peak (the P–S bond of the PS₄ tetrahedra, about 2.05 Å) and the first Li–S peak (about 2.4–2.5 Å) appear at the same distances as in DFT, so the fine-tuned model keeps the local structure. Differences in peak height come mostly from the short MLIP run (2 ps, few frames) and the slightly different cell volume.

*Your numbers may differ slightly from run to run.*

## Take-aways

- **uMLIP or fine-tuning?** A large uMLIP works out of the box, but it is slow and not better for every property. A small pretrained model fine-tuned on a few dozen DFT structures of your system is fast and matches your DFT setting.
- **Pretraining is what makes a few structures enough.** With the same 42 structures, training from scratch fails.
- **Check what fine-tuning changed.** Compare every error with its value before training (epoch 0). Here fine-tuning fixed the energy and the stress, but not the forces.
- **A low RMSE is not the end.** Check the properties you simulate (volume, energy–volume curve, RDF) against DFT, and make sure the fine-tuning data cover the conditions you want to simulate (volumes, temperatures, defects).

The post-class notebook (self-study) continues from here: Li diffusion and activation energy (A), how many structures fine-tuning needs and which training settings improve the forces (B), and whether the models can tell when they are unsure (C, optional).

## Submission

Run the next cell and submit the eight files. The cell also downloads `checkpoint_fine_tuned.pth`, your fine-tuned model: do not submit it, but keep it for the post-class notebook (it is lost when the Colab runtime ends).

1. `inclass_results.csv`: RMSE table of all models (Practice 1 and 2)
2. `inclass_volumes.csv`: relaxed volume per unit cell of each model (Practice 3)
3. `practice2_learning_curve.png`: learning curves, fine-tuned and from scratch
4. `practice2_rmse.png`: RMSE comparison
5. `practice3_relax.png`: structures before and after relaxation, and the volume during relaxation
6. `practice3_eos.png`: energy–volume curves of the models and DFT
7. `practice3_msd.png`: MSD and Li paths
8. `practice3_rdf.png`: radial distribution functions, MLIP MD and DFT MD

In [ ]:
summary.to_csv('inclass_results.csv')
pd.Series(volumes, name='Volume per unit cell (Å³)').round(1).to_csv('inclass_volumes.csv')
print(summary.round(3)); print(pd.Series(volumes).round(1))

try:
    from google.colab import files
    for f in ['inclass_results.csv', 'inclass_volumes.csv', 'practice2_learning_curve.png', 'practice2_rmse.png',
              'practice3_relax.png', 'practice3_eos.png', 'practice3_msd.png', 'practice3_rdf.png', 'checkpoint_fine_tuned.pth']:
        files.download(f)
except ImportError:
    pass